In [7]:
%pip install pandas 
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.
  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.2.0-py3-none-any.whl (45 kB)
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd

# 讀取 csv 檔案

df = pd.read_csv('./Sample-superstore.csv', encoding='latin1')
df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [11]:
import pymysql
from configparser import ConfigParser
from create_table import create_superstore_tables

# 讀取 .env 檔案取得資料庫連線資訊
config = ConfigParser()
config.read('../Chapter1/config.ini')

# 建立資料庫連線
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

with connection.cursor() as cursor:
    # 建立資料庫
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS superstore")

    cursor.execute(f"SHOW DATABASES")
    dbs = cursor.fetchall()
    print(dbs)

    # 建立資料表
    create_superstore_tables('superstore')
    

[{'Database': 'chapter2'}, {'Database': 'chapter3'}, {'Database': 'classicmodels'}, {'Database': 'information_schema'}, {'Database': 'my_database'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'superstore'}, {'Database': 'sys'}, {'Database': 'testdb'}, {'Database': 'transaction_test'}, {'Database': 'world'}]
{'Tables_in_superstore': 'customers'}
{'Tables_in_superstore': 'orderdetails'}
{'Tables_in_superstore': 'orders'}
{'Tables_in_superstore': 'products'}


In [12]:
# 建立資料庫連線
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database='superstore'
)

In [16]:
import pandas as pd

# 讀取 CSV 檔案
df = pd.read_csv('./Sample-superstore.csv', encoding='latin1')

# 提取客戶欄位的資料
customers = df[['Customer ID', 'Customer Name', 'Segment']]


# 去除重複的客戶資料
customers = customers.drop_duplicates()

customers

,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
2,DV-13045,Darrin Van Huff,Corporate
3,SO-20335,Sean O'Donnell,Consumer
5,BH-11710,Brosina Hoffman,Consumer
12,AA-10480,Andrew Allen,Consumer
...,...,...,...
8666,CJ-11875,Carl Jackson,Corporate
9209,RS-19870,Roy Skaria,Home Office
9399,SC-20845,Sung Chung,Consumer
9441,RE-19405,Ricardo Emerson,Consumer


Customers

In [39]:
# 取得 df 中的客戶資料 (去重複)
customers = df[['Customer ID', 'Customer Name', 'Segment']].drop_duplicates()

# 轉成 list 以便後續寫入
customers_list = customers.values.tolist()

# 寫入 Customers 資料表
with connection.cursor() as cursor:
    sql = """
INSERT INTO Customers (customer_id, customer_name, segment) VALUES (%s, %s, %s)
"""
    cursor.executemany(sql, customers_list)
    connection.commit()

IntegrityError: (1062, "Duplicate entry 'CG-12520' for key 'customers.PRIMARY'")

Orders 

In [35]:
from datetime import datetime

# 取得 df 中的訂單資料
orders = df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID']].drop_duplicates()


# 將日期格式轉換為 datetime.date
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format='%m/%d/%Y').dt.date
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format='%m/%d/%Y').dt.date

# 轉成 list 以便後續寫入
orders_list = orders.values.tolist()

# 寫入 Orders 資料表
with connection.cursor() as cursor:
    sql = """
INSERT INTO Orders (Order_ID, Order_Date, Ship_Date, Ship_Mode, Customer_ID) VALUES (%s, %s, %s, %s, %s)
"""
    cursor.executemany(sql, orders_list)
    connection.commit()
orders

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID
0,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520
2,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045
3,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335
5,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710
12,CA-2017-114412,2017-04-15,2017-04-20,Standard Class,AA-10480
...,...,...,...,...,...
9986,CA-2016-125794,2016-09-29,2016-10-03,Standard Class,ML-17410
9987,CA-2017-163629,2017-11-17,2017-11-21,Standard Class,RA-19885
9989,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400
9990,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060


Products

In [51]:
# 取得 df 中的產品資料
print(df.columns)
products = df[['Product ID', 'Product Name', 'Category', 'Sub-Category']].drop_duplicates()

# 轉成 list 以便後續寫入
products_list = products.values.tolist()


# 寫入 Products 資料表
with connection.cursor() as cursor: 
    sql = """
    INSERT INTO Products (product_ID, product_Name, category, sub_category) VALUES (%s, %s, %s, %s)
"""
    cursor.executemany(sql, products_list)
    connection.commit()



Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')


OrderDetails 不能去重複

In [54]:
# 取得 df 中的訂單明細資料
print(df.columns)
orderdetails_df = df[['Row ID', 'Order ID','Product ID', 'Sales', 'Quantity', 'Discount', 'Profit']]

# 轉成 list 以便後續寫入
orderdetails_list = orderdetails_df.values.tolist()

# 寫入 OrderDetails 資料表
with connection.cursor() as cursor: 
    sql = """
    INSERT INTO orderdetails (row_ID, order_ID, product_ID, sales, quantity, discount, profit) 
    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""
    cursor.executemany(sql, orderdetails_list)
    connection.commit()



Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')
